# 階段 3：Context 預算實驗（步驟 1、2）

對應文件：`docs/03-stage-harness-1.md`。這份 notebook 做「量測」的部分：一份上千筆的 Trivy 報告直接塞進 context 會怎樣、裁剪與壓縮各省多少。狀態檢查點、權限、hooks 是應用程式碼，照文件寫在 `src/vuln_agent/`。

**前置**：階段 2 的 agent loop 能跑；本機裝好 [Trivy](https://trivy.dev/)。

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent  # kernel 的工作目錄是 notebooks/，往上一層才是專案根目錄

## 步驟 1：準備報告、量出基準

在終端機掃**自己的** image（參數以 Trivy 官方文件為準）：

```bash
mkdir -p data/reports
trivy image --format json --output data/reports/trivy-myapp.json <你的 image>
```

想要多一點結果，可以用舊版 base image 另 build 一個練習用 image。

### 1.1 統計報告內容

Trivy JSON 的弱點在 `Results[].Vulnerabilities[]`，常用欄位：`VulnerabilityID`、`PkgName`、`InstalledVersion`、`FixedVersion`、`Severity`（以你實際拿到的檔案為準）。

In [ ]:
REPORT = ROOT / "data" / "reports" / "trivy-myapp.json"
raw = REPORT.read_text()
report = json.loads(raw)

vulns = [v for r in report.get("Results", []) for v in (r.get("Vulnerabilities") or [])]
print("總筆數:", len(vulns))
print("各 severity:", Counter(v.get("Severity") for v in vulns))
print("不重複 CVE:", len({v.get("VulnerabilityID") for v in vulns}))
print(f"有 FixedVersion 的比例: {sum(bool(v.get('FixedVersion')) for v in vulns) / max(len(vulns), 1):.0%}")
print("檔案大小:", f"{len(raw) / 1024:.0f} KB")

### 1.2 整份報告有多少 tokens？

用 `count_tokens` 計算，不會生成回答。太大可能直接報錯，報錯本身也是一個數據點，記下來。

In [ ]:
from vuln_agent.llm import client, DEFAULT_MODEL

try:
    n = client.messages.count_tokens(model=DEFAULT_MODEL, messages=[{"role": "user", "content": raw}])
    print("整份報告:", n.input_tokens, "tokens")
except Exception as e:
    print("超過上限或失敗:", e)

### 1.3 準備 5 題報告問題與標準答案

標準答案用程式算，不要用眼睛數。算好之後寫進 `evals/report.jsonl`。

In [ ]:
GOLD = [
    {"question": "有幾個 Critical？",
     "answer": sum(v.get("Severity") == "CRITICAL" for v in vulns)},
    {"question": "openssl 相關的有哪些、修到哪一版？",
     "answer": sorted({(v["VulnerabilityID"], v.get("FixedVersion")) for v in vulns if "openssl" in v.get("PkgName", "")})},
    # TODO：再寫 3 題，其中一題要跟 KEV 有關（需要查階段 2 的資料庫）
]

(ROOT / "evals").mkdir(exist_ok=True)
with open(ROOT / "evals" / "report.jsonl", "w") as f:
    for g in GOLD:
        f.write(json.dumps(g, ensure_ascii=False, default=list) + "\n")

### 1.4 天真版基準

在 agent 加一個最天真的 `read_scan_report(path)`：直接回傳整份 JSON。用它跑這 5 題，記錄每題的 input tokens、花費、延遲、答對與否。

下面的 `run_case` 請接上你階段 2 寫的 agent（例如 `from vuln_agent.agent import run_agent`），讓它回傳答案與用量。

In [ ]:
import time
from vuln_agent.llm import cost

def run_case(question: str, strategy: str) -> dict:
    t0 = time.time()
    # TODO：用指定策略（naive / trim / trim+compact）跑 agent，取得最後答案與累計 usage
    answer, usage = ..., ...
    return {"strategy": strategy, "question": question, "answer": answer,
            "input_tokens": usage.input_tokens, "cost": cost(usage), "seconds": round(time.time() - t0, 1)}

results = []
# for g in GOLD:
#     results.append(run_case(g["question"], "naive"))

## 步驟 2：裁剪、摘要、token 預算

### 2.1 通用的工具結果裁剪

任何工具結果超過預算就截斷，並且**告訴模型被截斷了**，讓它知道要用篩選條件縮小範圍。下面是可直接用的版本，用字元數粗估 token（中文約 1 字 1 token、英文約 4 字元 1 token），要精準就改呼叫 `count_tokens`，但每次都多一個 API 請求。

In [ ]:
def trim_to_budget(rows: list, max_tokens: int = 2000, chars_per_token: float = 3.0) -> dict:
    budget = int(max_tokens * chars_per_token)
    kept, used = [], 0
    for row in rows:
        size = len(json.dumps(row, ensure_ascii=False))
        if used + size > budget:
            break
        kept.append(row)
        used += size
    out = {"rows": kept, "total": len(rows)}
    if len(kept) < len(rows):
        out["note"] = f"已截斷：只回傳 {len(kept)} / {len(rows)} 筆，請用 severity、package 等篩選條件縮小範圍"
    return out

# 自我檢查
demo = trim_to_budget([{"id": i, "pad": "x" * 100} for i in range(1000)], max_tokens=500)
assert 0 < len(demo["rows"]) < 1000 and "note" in demo and demo["total"] == 1000
assert "note" not in trim_to_budget([{"id": 1}])
print(len(demo["rows"]), demo["note"])

### 2.2 對話層級的壓縮

**TODO**：寫 `compact(messages, max_tokens)`：每輪呼叫前估算 context 大小，超過門檻時，把較早的 `tool_result` 內容換成一行摘要（例如「[已摘要] query_findings 回傳 230 筆，前 3 筆是 ...」）。不能丟的：使用者原始問題、已確認的結論、最近 N 輪。

提示：只改 `tool_result` 的 `content`，不要刪掉整則訊息，否則 `tool_use` 和 `tool_result` 會配不起來，API 會報錯。

另外讀一下 API 內建的 context editing 與 compaction（官方文件「Context editing」「Compaction」），想想自己寫的版本和它們差在哪。

In [ ]:
def compact(messages: list, max_tokens: int = 20_000) -> list:
    ...  # TODO

### 2.3 三種策略比較

三種策略各跑一次 5 題，做成一張表。

In [ ]:
# for strategy in ("naive", "trim", "trim+compact"):
#     for g in GOLD:
#         results.append(run_case(g["question"], strategy))

def show(rows: list[dict], cols: list[str]):
    print(" | ".join(cols))
    print(" | ".join("---" for _ in cols))
    for r in rows:
        print(" | ".join(str(r.get(c, "")) for c in cols))

show(results, ["strategy", "question", "input_tokens", "cost", "seconds"])

**延伸實驗**：刻意把同一個 session 拉長到 20 輪（連續追問報告細節），每 5 輪問一題 `GOLD` 裡的題目，觀察答案品質從第幾輪開始下降、壓縮版有沒有撐比較久。

做完回到 `docs/03-stage-harness-1.md` 的步驟 3（狀態與檢查點）。